# Chapter 14: LangGraph 챗봇 구축 (LangGraph Chatbot)

LangGraph로 실용적인 AI 챗봇을 단계적으로 구축합니다.

**Sections:**
- 14.0 기본 챗봇 (MessagesState + init_chat_model)
- 14.1 도구 노드 (Tool Calling + ReAct 패턴)
- 14.2 메모리 (SQLite 체크포인터)
- 14.3 Human-in-the-loop (interrupt + Command resume)
- 14.4 타임 트래블 (State Fork)

---
## 14.0 Setup & Environment

In [29]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL_NAME:", os.getenv("OPENAI_MODEL_NAME"))

OPENAI_API_KEY: a703ad29...
OPENAI_BASE_URL: https://lgcns-assetization3-japan-east.openai.azure.com/openai/v1
OPENAI_MODEL_NAME: gpt-5.1


In [30]:
# 패키지 확인
from importlib.metadata import version

print(f"langgraph: {version('langgraph')}")
print(f"langchain: {version('langchain')}")

langgraph: 1.1.3
langchain: 1.2.13


---
## 14.0 기본 챗봇 — MessagesState

Chapter 13에서 `TypedDict`로 직접 상태를 정의했다면,
챗봇에서는 **`MessagesState`** 를 사용한다.

- `MessagesState`는 `messages: list` 필드를 기본 제공
- 메시지 **추가(append)** 동작이 기본 (리듀서 내장)
- `init_chat_model("openai:모델명")`로 LLM 초기화

In [31]:
import os
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# MessagesState 상속 — messages 필드 자동 포함
class State(MessagesState):
    pass


graph_builder = StateGraph(State)


def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile()

result = graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "how are you?"},
        ]
    }
)

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content[:100]}")

human: how are you?
ai: I don’t have feelings, but I’m working properly and ready to help. What can I do for you today?


### Exercise 14.0

1. `MessagesState` 대신 `TypedDict`로 직접 `messages: list`를 정의하면 어떤 차이가 생기는가?
2. `init_chat_model`의 프로바이더를 `"anthropic:claude-sonnet-4-20250514"` 등으로 바꿔 보라.
3. `State`에 `system_prompt: str` 필드를 추가하여 동적 시스템 프롬프트를 구현해 보라.

In [4]:
# Try it yourself


---
## 14.1 도구 노드 — Tool Calling + ReAct 패턴

LLM이 **외부 도구를 호출**하고, 결과를 받아 최종 응답을 생성하는 패턴:

```
START → chatbot → [tool_calls?] → tools → chatbot → ... → END
```

핵심 컴포넌트:
- `@tool` — Python 함수를 LLM 도구로 변환
- `ToolNode` — 도구 실행 담당 노드
- `tools_condition` — tool_calls 있으면 tools로, 없으면 END로

In [32]:
import os
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# 1단계: 도구 정의
@tool
def get_weather(city: str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny."


# 2단계: LLM에 도구 바인딩
llm_with_tools = llm.bind_tools(tools=[get_weather])


class State(MessagesState):
    pass


graph_builder = StateGraph(State)


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# 3단계: ToolNode 추가
tool_node = ToolNode(tools=[get_weather])

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")  # 도구 결과 → 다시 챗봇으로

graph = graph_builder.compile()

# 도구가 필요한 질문
result = graph.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in machupichu"}]}
)

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content[:120]}")

human: what is the weather in machupichu
ai: 
tool: The weather in Machupicchu is sunny.
ai: The weather in Machupicchu is currently sunny.


In [6]:
# 도구가 필요 없는 질문 — tools 노드를 거치지 않고 바로 END
result2 = graph.invoke(
    {"messages": [{"role": "user", "content": "hello, how are you?"}]}
)

for msg in result2["messages"]:
    print(f"{msg.type}: {msg.content[:120]}")

human: hello, how are you?
ai: I don’t have feelings, but I’m working properly and ready to help you with whatever you need.

What can I do for you tod


### Exercise 14.1

1. 도구를 하나 더 추가(예: `get_time`)하고, LLM이 상황에 맞게 적절한 도구를 선택하는지 확인하라.
2. 도구의 docstring을 변경하면 LLM의 도구 선택 행동이 어떻게 달라지는지 실험하라.
3. `tools_condition`을 커스텀 조건 함수로 교체해 보라.

In [7]:
# Try it yourself


---
## 14.2 메모리 — SQLite 체크포인터

기본 그래프는 `invoke()` 끝나면 상태가 사라진다.
**체크포인터**를 사용하면 대화 상태를 DB에 저장:

- `SqliteSaver` — SQLite 기반 체크포인터
- `thread_id` — 대화 세션 구분 식별자
- 같은 thread_id → 대화 이어감, 다른 thread_id → 새 대화

In [8]:
import os
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


@tool
def get_weather(city: str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny."


llm_with_tools = llm.bind_tools(tools=[get_weather])


class State(MessagesState):
    pass


graph_builder = StateGraph(State)


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


tool_node = ToolNode(tools=[get_weather])

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

# 체크포인터 연결!
conn = sqlite3.connect("memory.db", check_same_thread=False)
graph = graph_builder.compile(checkpointer=SqliteSaver(conn))

In [9]:
# 첫 번째 대화 — thread_id="1"
config = {"configurable": {"thread_id": "1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Alice."}]},
    config=config,
)
print(result["messages"][-1].content)

Hi Alice, welcome back. What would you like to do next?


In [10]:
# 같은 thread_id로 이어서 — 이전 대화 기억함!
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config,
)
print(result["messages"][-1].content)

Your name is Alice.


In [11]:
# 다른 thread_id → 새로운 대화 (이름 모름)
config2 = {"configurable": {"thread_id": "2"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config2,
)
print(result["messages"][-1].content)

You still haven’t told me your name in this conversation, so I don’t know it.  

If you’d like me to use it, you can say, “My name is …” and I’ll remember it for the rest of this chat.


In [12]:
# 상태 히스토리 조회
for state in graph.get_state_history(config):
    print(f"next: {state.next}, messages: {len(state.values.get('messages', []))}")

next: (), messages: 12
next: ('chatbot',), messages: 11
next: ('__start__',), messages: 10
next: (), messages: 10
next: ('chatbot',), messages: 9
next: ('__start__',), messages: 8
next: (), messages: 8
next: ('chatbot',), messages: 7
next: ('__start__',), messages: 6
next: (), messages: 6
next: ('chatbot',), messages: 5
next: ('__start__',), messages: 4
next: (), messages: 4
next: ('chatbot',), messages: 3
next: ('__start__',), messages: 2
next: (), messages: 2
next: ('chatbot',), messages: 1
next: ('__start__',), messages: 0


### Exercise 14.2

1. 동일한 `thread_id`로 여러 번 대화하여 메모리가 실제로 유지되는지 확인하라.
2. `stream_mode="updates"`로 스트리밍 실행을 해 보라: `async for event in graph.astream(...)`
3. `get_state_history()`의 결과를 분석하여 각 노드 실행 후 상태 변화를 추적하라.

In [13]:
# Try it yourself


---
## 14.3 Human-in-the-loop — 사람 개입

AI가 자동 처리하는 것이 아니라, 중간에 **사람의 피드백**을 받는 패턴:

1. `interrupt()` — 그래프 실행 일시 중단
2. 사용자가 피드백 제공
3. `Command(resume=...)` — 피드백을 전달하며 실행 재개

```
chatbot → tools(interrupt!) → [사람 피드백] → tools(resume) → chatbot → END
```

In [33]:
import os
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.types import interrupt, Command

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# 사람 피드백 도구 — interrupt()가 핵심!
@tool
def get_human_feedback(poem: str):
    """Asks the user for feedback on the poem.
    Use this before returning the final response."""
    feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")
    return feedback


llm_with_tools = llm.bind_tools(tools=[get_human_feedback])


class State(MessagesState):
    pass


graph_builder = StateGraph(State)


def chatbot(state: State):
    response = llm_with_tools.invoke(
        f"""
        You are an expert in making poems.
        Use the `get_human_feedback` tool to get feedback on your poem.
        Only after you receive positive feedback you can return the final poem.
        ALWAYS ASK FOR FEEDBACK FIRST.

        Here is the conversation history:
        {state['messages']}
    """
    )
    return {"messages": [response]}


tool_node = ToolNode(tools=[get_human_feedback])

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

conn = sqlite3.connect("memory_hitl.db", check_same_thread=False)
graph = graph_builder.compile(checkpointer=SqliteSaver(conn))

In [34]:
# 1단계: 시 작성 요청 — interrupt()에서 멈춤
config = {"configurable": {"thread_id": "poem_1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Please make a poem about Python code."}]},
    config=config,
)

# 상태 확인 — next가 ('tools',)이면 인터럽트 대기중
snapshot = graph.get_state(config)
print("Next:", snapshot.next)

Next: ('tools',)


In [35]:
# 2단계: 부정적 피드백 → 수정 요청
result = graph.invoke(
    Command(resume="It is too long! Make it shorter, 4 lines max."),
    config=config,
)

snapshot = graph.get_state(config)
print("Next:", snapshot.next)  # 또 인터럽트 대기중일 수 있음

Next: ('tools',)


In [36]:
# 3단계: 긍정적 피드백 → 최종 시 반환
result = graph.invoke(
    Command(resume="It looks great!"),
    config=config,
)

# 대화 전체 출력
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content[:150]}")
    print("---")

# 완료 확인
snapshot = graph.get_state(config)
print("\nNext:", snapshot.next)  # () 이면 완료

human: Please make a poem about Python code.
---
ai: 
---
tool: It is too long! Make it shorter, 4 lines max.
---
ai: 
---
tool: It looks great!
---
ai: 
---
human: Please make a poem about Python code.
---
ai: 
---
tool: It is too long! Make it shorter, 4 lines max.
---
ai: 
---
human: Please make a poem about Python code.
---
ai: 
---
tool: It is too long! Make it shorter, 4 lines max.
---
ai: 
---
tool: It looks great!
---
ai: Python hums in indented lines,  
Logic flows where whitespace shines;  
From `if` to `for`, our thoughts unfold—  
A quiet script of stories told.
---
human: Please make a poem about Python code.
---
ai: 
---
tool: It is too long! Make it shorter, 4 lines max.
---
ai: 
---
tool: It looks great!
---
ai: Python sings in indented lines,  
Logic blooms where whitespace shines;  
From `if` to `for`, our stories flow—  
A quiet script in midnight’s glow.
---

Next: ()


### Exercise 14.3

1. 부정적 피드백을 여러 번 연속으로 제공하면 LLM이 어떻게 반응하는지 관찰하라.
2. `Command(resume=...)` 에 딕셔너리(구조화된 데이터)를 전달해 보라.
3. 검토 → 승인 → 배포 파이프라인처럼 여러 인터럽트 포인트를 설계해 보라.

In [18]:
# Try it yourself


---
## 14.4 타임 트래블 — State Fork

체크포인터가 저장한 **상태 히스토리**를 활용하여,
과거 시점으로 돌아가 **분기(fork)** 실행:

- `get_state_history()` — 모든 스냅샷 시간순 조회
- `update_state()` — 과거 체크포인트 기반 새 분기 생성
- 디버깅, A/B 테스팅, 롤백에 활용

In [19]:
import os
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class State(MessagesState):
    pass


graph_builder = StateGraph(State)


def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

conn = sqlite3.connect("memory_tt.db", check_same_thread=False)
graph = graph_builder.compile(checkpointer=SqliteSaver(conn))

In [20]:
# 대화 시작
config = {"configurable": {"thread_id": "tt_1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "I live in Europe. My city is Valencia."}]},
    config=config,
)
print(result["messages"][-1].content[:200])

Here are some well-regarded places in Valencia, grouped by style. All are in or near the city center (Ciutat Vella / Ruzafa / Eixample) unless noted.

### Paella & Traditional Rice (must-do in Valenci


In [21]:
# 이어서 질문
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What are some good restaurants near me?"}]},
    config=config,
)
print("Valencia 기준 응답:")
print(result["messages"][-1].content[:300])

Valencia 기준 응답:
I don’t have your exact street or neighborhood, but I can suggest good options in key areas of Valencia. If you tell me your approximate location (e.g., “near Ruzafa”, “old town”, “City of Arts and Sciences”, “Benimaclet”), I’ll narrow it down to places within a short walk.

For now, here are strong


In [22]:
# 상태 히스토리 탐색
state_history = list(graph.get_state_history(config))

for i, snap in enumerate(state_history):
    msgs = snap.values.get("messages", [])
    print(f"Snapshot {i}: next={snap.next}, messages={len(msgs)}")

print(f"\n총 {len(state_history)}개 스냅샷")

Snapshot 0: next=(), messages=7
Snapshot 1: next=('chatbot',), messages=6
Snapshot 2: next=('__start__',), messages=5
Snapshot 3: next=(), messages=5
Snapshot 4: next=('chatbot',), messages=4
Snapshot 5: next=('__start__',), messages=3
Snapshot 6: next=('chatbot',), messages=3
Snapshot 7: next=('__start__',), messages=2
Snapshot 8: next=(), messages=2
Snapshot 9: next=('chatbot',), messages=1
Snapshot 10: next=('__start__',), messages=0

총 11개 스냅샷


In [23]:
# 포크: "Valencia" → "Zagreb"로 변경하여 새 분기 생성
# 첫 번째 대화 직후 스냅샷을 찾아 포크
from langchain_core.messages import HumanMessage

# 사용자가 도시를 말한 시점의 스냅샷 선택 (next가 ('chatbot',)인 것)
fork_point = None
for snap in state_history:
    if snap.next == ("chatbot",) and len(snap.values.get("messages", [])) == 1:
        fork_point = snap
        break

if fork_point:
    # 과거 메시지를 Zagreb로 수정하여 새 분기
    fork_config = fork_point.config
    graph.update_state(
        fork_config,
        {"messages": [HumanMessage(content="I live in Europe. My city is Zagreb.")]},
    )

    # 새 분기에서 실행
    result_fork = graph.invoke(None, config=fork_config)
    print("Zagreb 분기 응답:")
    print(result_fork["messages"][-1].content[:300])

Zagreb 분기 응답:
Thanks, that helps for context.

When you ask location-dependent questions (weather, taxes, health systems, local laws, language, time zones, etc.), I’ll tailor answers specifically to Valencia and Spain/Europe where relevant.

If you want, tell me what you’re mainly interested in (e.g., cost of liv


### Exercise 14.4

1. 더 긴 대화를 나눈 후 중간 시점으로 포크하여 다른 분기를 만들어 보라.
2. 포크한 분기에서 원래 분기와 다른 질문을 하여 결과를 비교해 보라.
3. 타임 트래블이 유용한 실무 시나리오를 생각해 보라 (A/B 테스팅, 디버깅, 롤백).

In [24]:
# Try it yourself


---
## Final Exercises — 종합 실습

### 과제 1: 다중 도구 챗봇 (★★☆)

`get_weather`, `get_time`, `get_news` 3개 도구를 가진 챗봇을 만들어라.
LLM이 질문에 따라 적절한 도구를 선택하는지 확인하라.

In [25]:
# 과제 1: 여기에 작성


### 과제 2: 대화 이력 유지 챗봇 (★★☆)

체크포인터로 메모리를 가진 챗봇을 만들고:
1. thread_id="a"로 3번 대화
2. thread_id="b"로 1번 대화
3. thread_id="a"로 돌아와서 이전 대화를 기억하는지 확인

In [26]:
# 과제 2: 여기에 작성


### 과제 3: 코드 리뷰 Human-in-the-loop (★★★)

코드를 생성하고 사람의 피드백을 받는 워크플로우:
1. LLM이 코드를 생성
2. `interrupt()`로 사람에게 코드 리뷰 요청
3. 피드백 기반으로 코드 수정 or 최종 확정

In [27]:
# 과제 3: 여기에 작성


### 과제 4: 타임 트래블 A/B 테스트 (★★★)

1. 챗봇에게 "Python vs JavaScript" 의견을 물어보라
2. 그 시점을 포크하여 "Python vs Rust"로 변경
3. 두 분기의 응답을 비교하라

In [28]:
# 과제 4: 여기에 작성
